# 5. Dimensional Engagement Prediction (Linear Baseline)

This notebook builds a **simple, interpretable linear model** to predict channel engagement from Graphiko dimensions.
It emphasizes **maximum dimensionality coverage** while keeping the model transparent enough to inspect per-dimension significance.

Outputs include:
- A unified feature table with human-readable channel labels.
- Per-channel performance summaries.
- Global and per-channel dimension significance tables.
- Charts for coefficients, uncertainty bands, and prediction quality.


This setup cell imports all required libraries and configures plotting defaults for consistent visual output across the notebook.


In [ ]:
# Core data and math libraries
import json
from pathlib import Path

import numpy as np
import pandas as pd

# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Modeling and inference libraries
import statsmodels.api as sm
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.model_selection import KFold

# Configure display + plotting aesthetics
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)
sns.set_theme(style="whitegrid", context="talk")


Define paths and helper functions to load Graphiko artifacts. The helpers enforce canonical label handling (`channel_name` as primary display label) and deterministic de-duplication when names collide.


In [ ]:
# Path configuration. Update these locations if your exports live elsewhere.
BASE_DIR = Path("/content/drive/MyDrive/Graphiko")
SEMANTIC_ENGAGEMENT_PATH = BASE_DIR / "analysis" / "semantic_engagement" / "latest" / "channel_semantic_engagement.csv"
CHANNEL_PROJECTION_PATH = BASE_DIR / "analysis" / "channel_clustering_projection" / "latest" / "channels_projection.json"


def make_unique_channel_labels(df: pd.DataFrame, id_col: str, name_col: str) -> pd.DataFrame:
    """Return a copy with deterministic unique names while preserving canonical names first."""
    out = df.copy()
    counts = out[name_col].value_counts()
    dupes = set(counts[counts > 1].index)
    out["channel_name"] = out.apply(
        lambda r: f"{r[name_col]} ({r[id_col]})" if r[name_col] in dupes else r[name_col], axis=1
    )
    return out


def zscore_columns(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    """Return z-scored copy for specified numeric columns."""
    out = df.copy()
    for c in cols:
        std = out[c].std(ddof=0)
        out[c] = 0.0 if std == 0 else (out[c] - out[c].mean()) / std
    return out


Load all available dimension artifacts, then assemble one wide feature matrix keyed by channel. Missing dimensions are retained and imputed later to maximize dimensionality.


In [ ]:
engagement_df = pd.read_csv(SEMANTIC_ENGAGEMENT_PATH)
with open(CHANNEL_PROJECTION_PATH, "r", encoding="utf-8") as f:
    projection_payload = json.load(f)
projection_df = pd.DataFrame(projection_payload)

rename_map = {"id": "channel_id", "name": "channel_name_raw", "engagement": "engagement_score"}
projection_df = projection_df.rename(columns={k: v for k, v in rename_map.items() if k in projection_df.columns})
engagement_df = engagement_df.rename(columns={k: v for k, v in rename_map.items() if k in engagement_df.columns})

if "channel_id" in engagement_df.columns and "channel_id" in projection_df.columns:
    wide_df = projection_df.merge(engagement_df, on="channel_id", how="outer", suffixes=("_proj", "_eng"))
else:
    wide_df = projection_df.merge(engagement_df, on="channel_name_raw", how="outer", suffixes=("_proj", "_eng"))

wide_df["channel_name_raw"] = wide_df.get("channel_name_raw_proj", pd.Series(index=wide_df.index)).fillna(
    wide_df.get("channel_name_raw_eng", pd.Series(index=wide_df.index))
)
wide_df = make_unique_channel_labels(wide_df, "channel_id", "channel_name_raw")

target_candidates = ["engagement_score", "semantic_engagement", "avg_engagement", "performance"]
target_col = next((c for c in target_candidates if c in wide_df.columns), None)
if target_col is None:
    raise ValueError(f"Could not find target column. Tried: {target_candidates}")

exclude_cols = {"channel_id", "channel_name", "channel_name_raw", target_col}
numeric_cols = [c for c in wide_df.columns if c not in exclude_cols and pd.api.types.is_numeric_dtype(wide_df[c])]

model_df = wide_df[["channel_id", "channel_name", target_col] + numeric_cols].copy()
print(f"Rows: {len(model_df):,} | Numeric dimensions: {len(numeric_cols):,} | Target: {target_col}")
model_df.head(10)


Inspect missingness and descriptive statistics before modeling. This helps validate dimensional coverage and interpretability assumptions.


In [ ]:
missing_tbl = model_df[numeric_cols].isna().mean().sort_values(ascending=False).rename("missing_rate").to_frame()
desc_tbl = model_df[numeric_cols + [target_col]].describe().T

display(missing_tbl.head(25))
display(desc_tbl)


Fit an OLS model on standardized dimensions to quantify coefficient significance. Then fit a regularized ridge model for robust cross-validated prediction.


In [ ]:
X_raw = model_df[numeric_cols].copy()
y = model_df[target_col].astype(float)

imputer = SimpleImputer(strategy="median")
X_imputed = pd.DataFrame(imputer.fit_transform(X_raw), columns=numeric_cols, index=model_df.index)
X_z = zscore_columns(X_imputed, numeric_cols)

X_ols = sm.add_constant(X_z)
ols_res = sm.OLS(y, X_ols, missing="drop").fit()

coef_tbl = (
    pd.DataFrame({
        "dimension": ols_res.params.index,
        "coef": ols_res.params.values,
        "p_value": ols_res.pvalues.values,
        "t_stat": ols_res.tvalues.values,
    })
    .query("dimension != 'const'")
    .assign(abs_coef=lambda d: d["coef"].abs())
    .sort_values(["p_value", "abs_coef"], ascending=[True, False])
)

display(coef_tbl.head(30))
print(ols_res.summary())


Run cross-validation for the predictive baseline and compute per-channel residual diagnostics for performance breakdown.


In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("ridge", Ridge(alpha=1.0)),
])

pred = np.zeros(len(model_df))
for train_idx, test_idx in kf.split(model_df):
    X_train, X_test = X_raw.iloc[train_idx], X_raw.iloc[test_idx]
    y_train = y.iloc[train_idx]
    pipe.fit(X_train, y_train)
    pred[test_idx] = pipe.predict(X_test)

perf_tbl = model_df[["channel_id", "channel_name", target_col]].copy()
perf_tbl["predicted_engagement"] = pred
perf_tbl["residual"] = perf_tbl[target_col] - perf_tbl["predicted_engagement"]
perf_tbl["abs_error"] = perf_tbl["residual"].abs()

print("CV R²:", round(r2_score(y, pred), 4))
print("CV MAE:", round(mean_absolute_error(y, pred), 4))
display(perf_tbl.sort_values("abs_error", ascending=False).head(25))


Visualize global significance and prediction quality with coefficient and scatter plots.


In [ ]:
top_dims = coef_tbl.head(20).copy()
fig, axes = plt.subplots(1, 2, figsize=(22, 8))

sns.barplot(data=top_dims, y="dimension", x="coef", ax=axes[0], palette="coolwarm")
axes[0].set_title("Top 20 OLS Coefficients")
axes[0].set_xlabel("Standardized coefficient")
axes[0].set_ylabel("Dimension")

sns.scatterplot(x=y, y=pred, ax=axes[1], s=80)
lims = [min(y.min(), pred.min()), max(y.max(), pred.max())]
axes[1].plot(lims, lims, linestyle="--")
axes[1].set_title("Actual vs Predicted Engagement")
axes[1].set_xlabel("Actual")
axes[1].set_ylabel("Predicted")

plt.tight_layout()
plt.show()


Compute a channel-specific contribution breakdown by multiplying standardized feature values with fitted OLS coefficients.


In [ ]:
beta = coef_tbl.set_index("dimension")["coef"]
contrib = X_z[beta.index].mul(beta, axis=1)
contrib.insert(0, "channel_name", model_df["channel_name"].values)
contrib.insert(0, "channel_id", model_df["channel_id"].values)

contrib_long = (
    contrib.melt(id_vars=["channel_id", "channel_name"], var_name="dimension", value_name="contribution")
    .assign(abs_contribution=lambda d: d["contribution"].abs())
    .sort_values(["channel_name", "abs_contribution"], ascending=[True, False])
)

top_k = 10
top_channel_breakdown = contrib_long.groupby(["channel_id", "channel_name"]).head(top_k)
display(top_channel_breakdown.head(50))


Render a sample channel breakdown chart so each channel’s strongest positive and negative dimensions are visually clear.


In [ ]:
sample_channel = top_channel_breakdown["channel_name"].iloc[0]
sample_df = top_channel_breakdown[top_channel_breakdown["channel_name"] == sample_channel].copy()

plt.figure(figsize=(12, 7))
sns.barplot(data=sample_df, y="dimension", x="contribution", palette="vlag")
plt.title(f"Top {top_k} Dimension Contributions for {sample_channel}")
plt.xlabel("Contribution to predicted engagement")
plt.ylabel("Dimension")
plt.tight_layout()
plt.show()
